# 第 1 周 · 第 1 天 —— 网站摘要器（Website Summarizer）

第一次用 Python 调用 LLM。本笔记本会：

1. 配置 OpenAI 客户端并加载 API Key
2. 发一次简单的 chat completion（「讲个笑话」）
3. 把真实网页抓成干净文本
4. 让模型用 Markdown 摘要该页面

> **前置条件：** 把 `.env.example` 复制为 `.env`，并填入你的 `OPENAI_API_KEY`。

---

## 1. 环境准备与导入


In [ ]:
# ========== 导入：本练习会用到的库与本地 scraper 工具 ==========

# 标准库 os：读环境变量
import os

# load_dotenv：从 .env 加载密钥到进程环境
from dotenv import load_dotenv
# Markdown / display：在笔记本里漂亮展示模型输出
from IPython.display import Markdown, display
# OpenAI：Chat Completions 客户端
from openai import OpenAI
# 同目录 scraper.py：抓网页 + 清洗 HTML（fetch_website_content / clean_html）
from scraper import clean_html, fetch_website_content


## 2. 加载 API 凭证

`load_dotenv` 会把本地 `.env` 里的键值对读进环境变量。若缺少 OpenAI Key，这里**立刻失败**（fail fast），避免后面调用到一半才报错。


In [ ]:
# ========== 读取并校验 OPENAI_API_KEY ==========

# override=True：.env 中的值覆盖进程里已有的同名环境变量
load_dotenv(override=True)
# 取出密钥字符串
api_key = os.getenv("OPENAI_API_KEY")

# 没有密钥就抛错，强制你先配置好再继续
if not api_key:
    raise ValueError("OPENAI_API_KEY environment variable not set")


## 3. 你的第一次 Chat Completion

Chat Completions API 接收一组 `messages`。每条消息有 `role`（`system` / `user` / `assistant`）和 `content`。这里只发一条 user 消息，然后打印模型回复。


In [ ]:
# ========== 构造第一条用户消息 ==========

# 发给模型的自然语言内容（保留英文：这是 prompt，会影响回答）
message = "Hi GPT it's my first time using you, can you tell me a joke?"
# messages 列表：目前只有一条 user；后面摘要任务会再加 system
messages = [
    {"role": "user", "content": message},
]


In [ ]:
# ========== 创建客户端并完成第一次对话 ==========

# 实例化客户端：会自动从环境变量读取 OPENAI_API_KEY，无需显式传入
openai = OpenAI()

# 发起一次非流式 Chat Completions
response = openai.chat.completions.create(
    # 模型 id 保留原样（影响计费与能力）
    model="gpt-5-nano",
    messages=messages,  # type: ignore
)

# 回复正文在 choices[0].message.content；包一层 Markdown 显示
display(Markdown(f"**GPT:** {response.choices[0].message.content}"))


## 4. 抓取网站

摘要之前，需要先把页面变成纯文本。  
`fetch_website_content`（定义在 [scraper.py](scraper.py)）会下载页面、去掉 script/style/图片等，并返回标题 + 可见文本。


In [ ]:
# ========== 抓取个人站点并打印清洗后的文本 ==========

# url 指向要摘要的页面；返回值通常是标题+正文拼好的字符串
my_website_content = fetch_website_content(url="https://azam-sys.netlify.app/")
# 先肉眼看一眼抓到了什么，便于调试 token / 噪声
print(my_website_content)


## 5. 用 LLM 摘要页面

这是经典的 **system + user prompt** 模式：

- **system prompt**：定模型角色与输出格式
- **user prompt**：携带实际数据（抓到的页面文本）


In [ ]:
# ========== system prompt：角色 + 输出格式（纯字符串，无插值） ==========

# 发给模型的指令保留英文；改译会改变摘要风格/格式
system_prompt = (
    "You are a helpful assistant that summarizes website content. "
    "Provide a concise summary of the main points and topics covered on the "
    "website. Respond in markdown format."
)


In [ ]:
# ========== user prompt：把抓到的页面正文塞给模型 ==========

# f-string 插值：把 my_website_content 嵌进提示词
user_prompt = f"Here is the content of the website: {my_website_content}"


In [ ]:
# ========== 封装：把已拼好的 messages 发给模型，返回回复文本 ==========

# Dict / List：给 messages 参数做类型标注，方便阅读与静态检查
from typing import Dict, List


def summarize_website_content(messages: List[Dict[str, str]]) -> str | None:
    """Send a prepared messages list to the model and return the reply text.

    Args:
        messages: Chat messages (system + user) describing the task and data.

    Returns:
        The model's response content, or ``None`` if the model returned no text.
    """
    # 调用 Chat Completions；messages 已在外部拼好
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=messages,  # type: ignore
    )
    # 只取助手文本；可能为 None（极少见）
    return response.choices[0].message.content


In [ ]:
# ========== 把 system + user 拼成 API 期望的 messages 列表 ==========

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]


In [ ]:
# ========== 调用摘要函数并渲染结果 ==========

# 传入刚拼好的 messages，拿到摘要字符串
response = summarize_website_content(messages)

# 前缀 **GPT:** 只是展示标签；正文按 Markdown 渲染
display(Markdown(data=f"**GPT:** {response}"))


## 6. 复用助手函数摘要另一页

既然 `summarize_website_content` 已是函数，换站点只需：抓取 → 拼 messages → 再调一次助手。


In [ ]:
# ========== 换一个 URL：CNN 首页 ==========

# 抓取新页面文本
content = fetch_website_content(url="https://cnn.com/")

# 复用同一个 system_prompt；user 侧换成新 content
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": f"Here is the content of the website: {content}"},
]


In [ ]:
# ========== 对 CNN 页面调用同一摘要函数 ==========

summary = summarize_website_content(messages)
display(Markdown(data=f"**GPT:** {summary}"))


## 7. 基于 Selenium 的网页抓取

基于 `requests` 的抓取器往往只能看到**初始 HTML**。对靠 JavaScript 渲染内容的站点（SPA、无限滚动等），需要真正的浏览器。**Selenium** 驱动真实 Chrome，拿到渲染后的页面，再用 `clean_html` 压低 token。

下面的 `WebsiteSummarizer` 把「抓取 → 摘要 → 展示」捆成一个小对象。

> **依赖：** `selenium`（已在 `requirements.txt`）。Selenium 4.6+ 会自动下载匹配的 chromedriver，一般无需手装驱动——本机装好 Chrome 即可。  
> 加载时可能弹出浏览器窗口；若要无界面，可自行加 `options.add_argument("--headless=new")`。


In [ ]:
# ========== WebsiteSummarizer：Selenium 抓取 + LLM 摘要 ==========

# webdriver：启动/控制浏览器；Options：Chrome 启动参数
from selenium import webdriver
from selenium.webdriver.chrome.options import Options


class WebsiteSummarizer:
    """Fetch a JS-rendered page with Selenium and summarize it with an LLM.

    Typical usage:
        ws = WebsiteSummarizer(url)
        ws.selenium_fetch_content()
        ws.summarize_content()
        ws.display_summary()
    """

    def __init__(self, url: str, base_url: str = "https://api.openai.com/v1", api_key: str | None = None, model: str = "gpt-5-nano") -> None:
        # 要抓取的目标地址
        self.url = url
        # 清洗后的页面文本（尚未抓取时为 None）
        self.html: str | None = None 
        # 模型摘要结果（尚未生成时为 None）
        self.summary: str | None = None 
        # OpenAI（或兼容）客户端：可用 base_url / api_key 指向不同供应商
        self.client = OpenAI(base_url=base_url, api_key=api_key)
        # 本实例使用的模型名
        self.model = model

    def selenium_fetch_content(self) -> None:
        """Load the page in headless Chrome and capture the cleaned text."""
        # 创建 Chrome 选项对象
        options = Options()
        # 自定义 User-Agent，降低被站点当机器人拦下的概率
        options.add_argument(
            "User-Agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
        )
        # 启动 Chrome 实例
        driver = webdriver.Chrome(options=options)
        try:
            # 打开目标 URL，等待页面加载
            driver.get(url=self.url)
            # 隐式等待：找元素时最多等 5 秒（给 JS 一点渲染时间）
            driver.implicitly_wait(5)
            # 把渲染后的 HTML 洗成标题+可见文本，显著减少发给模型的 token
            self.html = clean_html(html=driver.page_source)
        except Exception as e:
            # 错误文案保留英文（原逻辑）
            print(f"Error fetching content with Selenium: {e}")
        finally:
            # 无论成功失败都关掉浏览器，避免僵尸进程
            driver.quit()

    def summarize_content(self) -> None:
        """Summarize the fetched content using the LLM."""
        # 还没抓到内容就摘要：直接报错，提示先调 selenium_fetch_content
        if not self.html:
            raise ValueError(
                "Content not fetched yet. Call selenium_fetch_content() first."
            )

        # system：角色与 Markdown 输出要求（英文 prompt 不翻译）
        system_prompt = (
            "You are a helpful assistant that summarizes website content. "
            "Provide a concise summary of the main points and topics covered on the "
            "website. Respond in markdown format."
        )
        # user：塞入清洗后的页面文本
        user_prompt = f"Here is the content of the website: {self.html}"

        # 拼 messages
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        # 用实例自己的 client / model 发起请求
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,  # type: ignore
        )
        # 存到实例属性，供 display_summary 使用
        self.summary = response.choices[0].message.content

    def display_summary(self) -> None:
        """Render the summary as markdown."""
        # 还没摘要就展示：提示先调 summarize_content
        if not self.summary:
            raise ValueError(
                "Summary not generated yet. Call summarize_content() first."
            )
        # 在笔记本中渲染 Markdown
        display(Markdown(data=self.summary))


In [ ]:
# ========== 端到端演示：Netflix 首页 ==========

# 创建摘要器，目标 URL 为 Netflix
ws = WebsiteSummarizer(url="https://www.netflix.com/")
# 1) Selenium 打开页面并清洗 HTML
ws.selenium_fetch_content()
# 2) 调 LLM 生成摘要
ws.summarize_content()
# 3) 在笔记本里显示 Markdown
ws.display_summary()


## 8. 把摘要器指向任意 OpenAI 兼容供应商

`WebsiteSummarizer` 已支持 `base_url`、`api_key`、`model`，因此同一套「抓取 → 摘要 → 展示」流程可以跑在 **Ollama 本地模型**（或任意 OpenAI 兼容端点）上——不用改类内部代码，只换构造参数。


In [ ]:
# ========== 同一流程，后端换成本地 Ollama ==========

# base_url 指向本机 Ollama 的 OpenAI 兼容口；api_key 占位；model 用已 pull 的本地名
ws = WebsiteSummarizer(url="https://dhan.co", base_url="http://localhost:11434/v1", api_key="ollama", model="llama3.2:3b")
# 抓取（Selenium）
ws.selenium_fetch_content()
# 用本地模型摘要
ws.summarize_content()
# 展示
ws.display_summary()
